# Predict a finished batch's quality from the historical boxes

**Not a trained model - a lookup.** `search()` builds boxes (X range + mean Y + Cp).
`predict.score(res, x)` finds which boxes a finished batch falls into and returns the
expected Viscosity / Kappa. See `CONCEPT.md` for the idea and the backtest result.

Predict **after discharge** (all X final); the lab Y is still hours away, so it is a
genuine look-ahead. No PI here - X comes from you (dict / csv).

In [ ]:
import importlib
import pandas as pd

import config, golden, capability as cap, predict
for mod in (config, golden, cap, predict):
    importlib.reload(mod)

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

df = golden.load_golden()

## 1. One batch -> which boxes, which Y?
Edit `x` to the finished batch's X. Keys can be column names or short labels.

In [ ]:
x = {
    "pfactor_overshoot": 40,
    "alkali_accumulator1_conc_mean_hot_liquor_filling": 95,
    "prehydrolysis_minutes": 70,
    "chip_weight_max_chip_filling": 36,
    "digester": 3,
    "pfactor_temp_setpoint_last_prehydrolysis": 165,
}

scored, cons = predict.predict(x)          # build res (same-digester) + score
display(scored)
print("consensus:", cons)

## 2. All boxes it enters (digester as an X too)
Score against every box, not just same-digester - shows all windows the batch sits in.

In [ ]:
res_all = cap.search(df)
scored_all = predict.score(res_all, x)
display(scored_all)
print("consensus:", predict.consensus(scored_all))

## 3. Backtest - is the lookup better than guessing the mean?
Leave-one-out over golden. Beats the baseline MAE = the method carries real signal.
(Slow: rebuilds res per batch.)

In [ ]:
predict.backtest(by_digester=True)